# Week 3 — Introduction to scikit-learn I
## From orders to a justified prediction
**Campus Corner · 12 November 2026 · 90 minutes**

**Guiding question:** *Can we predict an order's packing time before packing begins, and how should we choose a model?*

We will assemble a dataset, separate training, validation, and test data, compare a baseline with two regressions, and use the selected model for new orders. All orders are simulated; they are not evidence about a real business.

Work from top to bottom. Complete **Your turn** cells before running dependent cells. Plotting helpers and data-generation code are provided: you do not need to write them from memory. Use the [AI learning tutor](AI_learning_tutor_prompt.txt) for hints, debugging, and checking your reasoning.

## Learning goals and warm-up (0–10 min)
By the end, you can:
- define what one observation represents and select inputs available at prediction time,
- construct a numerical target and check basic data quality,
- explain and create a train/validation/test split,
- compare models with a training-only baseline and validation MAE,
- interpret coloured prediction plots and justify a selection,
- evaluate the final model on untouched test data,
- predict new orders and recognise a concrete extrapolation problem.

Our four recurring questions:
1. *What is our question?*
2. *What is our baseline?*
3. *How do we evaluate the result?*
4. *What can we conclude?*

**Warm-up:** What is a DataFrame? What does `.head()` show? In `company.average_sales()`, which part is the object and which part is the method?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

### Our observations and prediction time
One row represents one completed order. We want to predict the minutes needed to pack a new order **before packing starts**.

| Column | Meaning | Unit |
|---|---|---|
| `order_id` | Unique identifier | None |
| `item_count` | Total number of items, including fragile items | Items |
| `fragile_count` | Number of fragile items within the total | Items |
| `packing_start_min` | Actual start, on an arbitrary clock | Minutes |
| `packing_end_min` | Actual finish, on the same clock | Minutes |

The clock values below are synthetic bookkeeping, not a chronological time series. We assume independent orders from the same stable process.

**Before running:** Which columns could be available for a new order? Which information is only known afterwards?

### Provided setup: reproducible orders
Run this cell unchanged. As in Week 2, a seed makes the data reproducible. `integers(3, 21)` generates counts from 3 through 20. The binomial draw marks each item fragile with probability 0.2, so fragile items cannot outnumber all items.

The formula deliberately creates a simple linear world with random variation. Reading it is allowed; our models must still learn their coefficients from training data. We will interpret the formula in the recap.

In [ ]:
rng = np.random.default_rng(42)
n_orders = 300

item_count = rng.integers(3, 21, size=n_orders)
fragile_count = rng.binomial(n=item_count, p=0.2)
noise = rng.normal(loc=0, scale=1, size=n_orders)
packing_time = 2 + 0.5 * item_count + 1.5 * fragile_count + noise

packing_start = np.arange(n_orders) * 30.0
raw_orders = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "item_count": item_count,
    "fragile_count": fragile_count,
    "packing_start_min": packing_start,
    "packing_end_min": packing_start + packing_time,
})
raw_orders.head()

## Worked examples and Your turn — Assemble the dataset (10–30 min)
### Worked example: column selection and basic checks
`data["column"]` selects one column as a Series. `data[["column"]]` keeps a two-dimensional table. scikit-learn expects its feature input `X` to be a table, even for one feature.

The target `y` contains one numerical value per order. Predicting a numerical quantity is called **regression**.

In [ ]:
print("Rows and columns:", raw_orders.shape)
print(raw_orders.dtypes)
print("\nMissing values per column:")
print(raw_orders.isna().sum())
print("\nRepeated order IDs:", raw_orders["order_id"].duplicated().sum())

one_column_table = raw_orders[["item_count"]]
one_column_table.head()

### Your turn 1: construct the target
Copy the table, then create `packing_time_min` by subtracting the actual start from the actual end. Display the first rows.

**Hint:** A calculation between two columns works row by row. Do not subtract in the opposite direction.

In [ ]:
orders = raw_orders.copy()
# TODO: Create orders["packing_time_min"] from the two clock columns.
# orders["packing_time_min"] = ...
# orders.head()

### Check that the assembled rows make sense
Run the checks after completing Your turn 1. `.all()` asks whether every row passes a condition.

A missing target cannot be used for supervised training. A repeated order ID may indicate the same observation was included twice; identical item counts alone do not imply duplicate orders. Investigate suspicious records rather than silently replacing or deleting them.

If units were mixed (seconds versus minutes), convert them consistently before constructing the target. Our provided table is complete and already uses minutes.

In [ ]:
print("Every duration is positive:",
      (orders["packing_time_min"] > 0).all())
print("Fragile counts are non-negative:",
      (orders["fragile_count"] >= 0).all())
print("Fragile counts do not exceed item counts:",
      (orders["fragile_count"] <= orders["item_count"]).all())

### Your turn 2: choose columns and observations
Discuss with a neighbour, then write short answers.

1. Why is `packing_time_min` our target?
2. Which two columns are plausible inputs available before packing begins?
3. Why would `packing_end_min` or the duration itself be unsuitable inputs?
4. Why should we exclude the sequential `order_id` from this example?
5. If the same order appeared twice and the copies entered different data splits, what would go wrong?
6. Would these data support predictions for a different warehouse with a different packing process?

**Data leakage** means giving a model or selection procedure information it would not legitimately have. A feature that reveals the answer is one example.

**Your answers:**  
1. ...  
2. ...  
3. ...  
4. ...  
5. ...  
6. ...

### Our candidate feature sets
We will compare two pre-specified variants of the same regression:
- **Items only:** total item count.
- **Items + fragile:** total item count and fragile item count.

The extra feature has to justify its use through validation performance. The following lists make column selection explicit.

In [ ]:
features_one = ["item_count"]
features_two = ["item_count", "fragile_count"]
target = "packing_time_min"

## Worked example and Your turn — Train / validation / test (30–45 min)
Split the observations **before exploring relationships or fitting models**.

| Part | Job | May it influence model selection? |
|---|---|---|
| Training: 60% | Learn coefficients and the baseline constant | Yes |
| Validation: 20% | Compare the pre-specified candidates | Yes |
| Test: 20% | Evaluate the completed selection procedure | No |

First reserve 20% for testing. Then reserve **25% of the remaining 80%** for validation: `0.25 × 0.80 = 0.20`.

We split whole rows so inputs and targets remain aligned. A fixed `random_state` reproduces the split; it does not improve a model. Do not search for a seed giving a favourable result.

A random split is appropriate for our independent, simulated orders. For future-versus-past prediction use a time-aware split; repeated customers or other related units may require a grouped split. 60/20/20 is an example, not a universal rule.

In [ ]:
development, test = train_test_split(
    orders, test_size=0.20, random_state=42
)
train, validation = train_test_split(
    development, test_size=0.25, random_state=42
)

print("Training orders:", len(train))
print("Validation orders:", len(validation))
print("Test orders:", len(test))

### Your turn 3: explain the split
1. Predict the three row counts before reading the output.
2. Why would a second split with `test_size=0.20` not give 60/20/20 overall?
3. Which part learns coefficients? Which part chooses the candidate? Which part is held back?
4. Could you justify choosing a model because its **training** error is smallest?

**Your answers:** ...

### Explore training data only
Run this provided chart. Colour represents fragile item count. At similar total item counts, do orders with more fragile items tend to take longer?

Some points overlap because counts are discrete. We are looking for a broad pattern, not a perfect rule.

In [ ]:
plt.figure(figsize=(7, 4))
points = plt.scatter(
    train["item_count"], train[target],
    c=train["fragile_count"], cmap="viridis", alpha=0.8
)
plt.colorbar(points, label="Fragile items")
plt.xlabel("Total items (including fragile items)")
plt.ylabel("Actual packing time (minutes)")
plt.title("Training orders: possible predictive information")
plt.tight_layout()
plt.show()

## Worked example — Baseline and selection criterion (45–55 min)
### What is our baseline?
Predict the **median training packing time** for every order. This ignores all features and gives a useful reference for whether a more elaborate model helps. The median is a sensible constant for MAE: it minimises the sum of absolute errors on the training observations.

### How do we evaluate the result?
Our primary selection criterion is **validation mean absolute error (MAE)**. Lower is better:

**MAE = average of |predicted time − actual time|.**

An MAE of 2 minutes means the predictions differ from actual times by 2 minutes on average in that evaluated sample. It is neither a maximum error nor an uncertainty interval for each order.

**Quick hand calculation:** Actual times are 5, 8, 11 minutes; predictions are 6, 6, 14. Calculate the three absolute errors and their average before running the next cell.

In [ ]:
example_actual = np.array([5, 8, 11])
example_predictions = np.array([6, 6, 14])
print("Absolute errors:", np.abs(example_predictions - example_actual))
print("MAE:", mean_absolute_error(example_actual, example_predictions))

### Create → fit → predict → evaluate
Remember objects and methods from Week 2:
- `DummyRegressor(...)` creates an object.
- `.fit(X_train, y_train)` learns from training observations.
- `.predict(X_validation)` produces one prediction per row; it receives no target values.
- `mean_absolute_error(y_validation, predictions)` compares predictions with held-out actual values.

We pass a feature table to the baseline for a consistent interface, although this baseline ignores the feature values.

In [ ]:
X_train_one = train[features_one]
X_validation_one = validation[features_one]
y_train = train[target]
y_validation = validation[target]

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train_one, y_train)
prediction_baseline = baseline.predict(X_validation_one)
mae_baseline = mean_absolute_error(y_validation, prediction_baseline)

print(f"Training median: {y_train.median():.2f} minutes")
print(f"Baseline validation MAE: {mae_baseline:.2f} minutes")

## Worked example and Your turn — Compare and select (55–72 min)
### Worked example: regression with one feature
A line predicts packing time from total item count:

`predicted time = intercept + coefficient × item_count`.

`LinearRegression` learns these values from training data. Its fitting procedure minimises squared training errors. Our **evaluation and selection metric** is still MAE; the training objective and selection metric need not be identical.

This first model uses default ordinary least squares. Hyperparameter search comes next week.

In [ ]:
model_one = LinearRegression()
model_one.fit(X_train_one, y_train)
prediction_one = model_one.predict(X_validation_one)
mae_one = mean_absolute_error(y_validation, prediction_one)

print(f"Intercept: {model_one.intercept_:.2f} minutes")
print(f"Item coefficient: {model_one.coef_[0]:.2f} minutes per item")
print(f"Items-only validation MAE: {mae_one:.2f} minutes")

### Your turn 4: add the second feature
Use the worked example to:
1. create `X_train_two` and `X_validation_two` using `features_two`,
2. create and fit `model_two`,
3. produce `prediction_two`,
4. calculate `mae_two` on the **same validation orders**.

This model predicts `intercept + coefficient_1 × item_count + coefficient_2 × fragile_count`.

**Hint:** Use the same `y_train` and `y_validation`. Do not call `.fit()` on validation data. Complete this cell before running the comparison.

In [ ]:
# TODO: Replace each ... and uncomment the lines.
# X_train_two = ...
# X_validation_two = ...
# model_two = LinearRegression()
# model_two.fit(..., ...)
# prediction_two = model_two.predict(...)
# mae_two = mean_absolute_error(..., ...)
# print(f"Two-feature validation MAE: {mae_two:.2f} minutes")

### Compare the candidates
Use the unrounded values for selection. Our rule, set before examining results: select the smallest validation MAE among the baseline and both regressions. In an exact tie, prefer the simpler candidate (baseline, then one feature, then two).

Also report whether the improvement is substantial enough to justify collecting the extra feature. A small difference on one validation split is not proof of a robust improvement.

In [ ]:
comparison = pd.DataFrame({
    "candidate": ["Median baseline", "Items only", "Items + fragile"],
    "validation_MAE_minutes": [mae_baseline, mae_one, mae_two],
})
comparison

### Three models, the same validation orders
Each point is an order:
- x = its actual packing time; y = its predicted packing time.
- The diagonal marks perfect predictions.
- Below the diagonal means **underprediction**; above means **overprediction**.
- Colour encodes fragile item count, with the same scale in every panel.
- All panels use identical limits and equal axis scales.

The vertical distance to the diagonal is the signed error. MAE averages its absolute magnitude. These plots work with any number of input features because the axes show the target and prediction, not the features.

**Provided plotting helper:** Run the next cell unchanged. Read its output; mastering the plotting code is optional.

In [ ]:
def plot_predictions(actual, predictions, names, fragile, split_label):
    # Shared limits and colours make panels directly comparable.
    all_values = np.concatenate([np.asarray(actual)] + predictions)
    lower = min(0, all_values.min() - 1)
    upper = all_values.max() + 1
    colour_scale = plt.Normalize(vmin=0, vmax=max(1, fragile.max()))
    fig, axes = plt.subplots(
        1, len(predictions), figsize=(5 * len(predictions), 4.6),
        sharex=True, sharey=True, squeeze=False, layout="constrained"
    )
    for ax, predicted, name in zip(axes[0], predictions, names):
        points = ax.scatter(
            actual, predicted, c=fragile, cmap="viridis",
            norm=colour_scale, alpha=0.85, edgecolors="white", linewidths=0.4
        )
        ax.plot([lower, upper], [lower, upper], "--", color="grey")
        ax.set_xlim(lower, upper)
        ax.set_ylim(lower, upper)
        ax.set_aspect("equal", adjustable="box")
        error = mean_absolute_error(actual, predicted)
        ax.set_title(f"{name}\n{split_label} MAE = {error:.2f} min")
        ax.set_xlabel("Actual packing time (minutes)")
        ax.grid(alpha=0.15)
    axes[0, 0].set_ylabel("Predicted packing time (minutes)")
    fig.colorbar(points, ax=list(axes[0]), label="Fragile items", shrink=0.8)
    plt.show()

plot_predictions(
    y_validation,
    [prediction_baseline, prediction_one, prediction_two],
    ["Median baseline", "Items only", "Items + fragile"],
    validation["fragile_count"],
    "Validation"
)

### Your turn 5: make and record your choice
1. Why do the baseline points form a horizontal line?
2. Which model has the lowest validation MAE? How much does it improve on the baseline and the other regression?
3. In the items-only plot, do colours suggest a pattern of under- or overprediction? Does the second feature help?
4. Choose a candidate using our rule. Explain the practical value and cost of the additional feature.
5. Why have we not looked at test performance yet?

Record your reasoning **before opening the test section**.

**Selection record:**  
My chosen candidate is ...  
Its validation MAE is ... minutes, compared with ...  
The extra feature is / is not worth collecting because ...  
One limitation of this comparison is ...

In [ ]:
# Record the winning row number from the comparison table: 0, 1, or 2.
# TODO: Replace None with your choice.
selected_index = None

## Interpretation — Final fit and untouched test (72–80 min)
We have now fixed the candidate and feature set. Refit **that choice** using training and validation together, so all development observations contribute to the final fitted parameters. The code below also supports choosing the baseline.

Then evaluate on test data once. Do not return to candidate selection because you dislike the test score. Re-running the same notebook for reproducibility is fine; repeatedly changing choices in response to test scores is not.

The test MAE estimates performance on new orders from this same assumed process. It does not establish performance in a different warehouse or for much larger orders.

In [ ]:
if selected_index not in [0, 1, 2]:
    raise ValueError("Complete your selection record and set selected_index to 0, 1, or 2.")

candidate_features = [features_one, features_one, features_two]
final_features = candidate_features[selected_index]
final_name = comparison.loc[selected_index, "candidate"]

if selected_index == 0:
    final_model = DummyRegressor(strategy="median")
else:
    final_model = LinearRegression()

final_model.fit(development[final_features], development[target])
test_predictions = final_model.predict(test[final_features])
test_mae = mean_absolute_error(test[target], test_predictions)

print("Final candidate:", final_name)
print("Final fitting rows:", len(development))
print(f"Final test MAE: {test_mae:.2f} minutes")

plot_predictions(
    test[target], [test_predictions], [final_name],
    test["fragile_count"], "Test"
)

### Interpretation
Complete these sentences:
- On the test orders, our selected model differs from the actual packing time by ... minutes on average.
- This does **not** mean that every prediction ...
- Validation and test MAE may differ because ...
- Our result is limited to ...

**Your interpretation:** ...

## Your turn — Predict a new order (80–88 min)
### Ordinary new orders
Compare A and B: only fragile item count changes. Compare B and C: only total item count changes.

**Think first:** Rank the expected packing times. Explain your reasoning. Then complete the prediction call below, and add one valid order of your own.

The prediction table must use the same feature names and order as the final fitting table. `final_features` takes care of this. Fragile items are included in total items.

In [ ]:
new_orders = pd.DataFrame({
    "item_count": [5, 5, 15],
    "fragile_count": [0, 3, 3],
}, index=["A", "B", "C"])

# TODO: Predict with final_model using new_orders[final_features].
# new_predictions = ...
# new_orders["predicted_minutes"] = new_predictions
# new_orders

**Discuss:** Can you calculate a MAE for these new orders? What information would you need?

**Your answer:** ...

### Focused challenge: a very large order
A new order contains **100 items, including 20 fragile items**. Run the provided prediction, then compare this order with the data used to fit the final model.

1. *How does this order compare with the orders in the training data?*
2. *The model returns a prediction. Does that mean it is reliable?*
3. *What might change when packing such a large order?*

Using a learned relationship beyond the observed input range is called **extrapolation**.

In [ ]:
large_order = pd.DataFrame({
    "item_count": [100],
    "fragile_count": [20],
})
large_prediction = final_model.predict(large_order[final_features])
print(f"Predicted packing time: {large_prediction[0]:.2f} minutes")

print("\nInput ranges in the final fitting data:")
print(development[features_two].agg(["min", "max"]))

**Your judgement:** ...

## Recap and outlook (88–90 min)
Answer our four questions in one sentence each:
1. **Question:** What did we predict, and at what point in time?
2. **Baseline:** What did the simple reference predict, and where did that value come from?
3. **Evaluation:** Which metric and which data part served each decision?
4. **Conclusion:** What improved, and what remains uncertain?

### Reveal the simulation's meaning
Our artificial world uses:

**packing time = 2 + 0.5 × total items + 1.5 × fragile items + noise**

The noise is independent normal variation with mean 0 and standard deviation 1 minute. Fragile items already count toward total items, so one fragile item contributes 0.5 + 1.5 = 2 minutes to expected effort. At 10 items, including 2 fragile items, the expected time is 10 minutes.

We deliberately made linear regression suitable. Even an appropriate model cannot predict independent random noise. The normal distribution is unbounded, so positive durations are not mathematically guaranteed; our concrete generated dataset was checked.

**Next week:** Keep this workflow and practise hyperparameter search with a neural network. Validation still guides choices; test data remain reserved for final evaluation.

### Before you leave
Save, restart the kernel, and run your completed core cells from top to bottom. Download your notebook before closing Binder. Record any AI help you used and how you checked its suggestions.

## Optional practice
These activities are outside the 90-minute core. They use the original validation results for understanding; they do not reopen the final test-based decision.

### A. Inspect signed errors
The following plots show **prediction − actual time** against total item count, coloured by fragile item count. A negative error means underprediction.

Before running, predict which model may have more visible colour patterns. Afterwards describe one pattern and one limitation. The MAE uses absolute errors, so positive and negative errors cannot cancel.

In [ ]:
validation_predictions = [prediction_baseline, prediction_one, prediction_two]
candidate_names = ["Median baseline", "Items only", "Items + fragile"]
errors = [p - y_validation.to_numpy() for p in validation_predictions]
error_limit = max(np.max(np.abs(e)) for e in errors) + 0.5
colour_scale = plt.Normalize(
    vmin=0, vmax=max(1, validation["fragile_count"].max())
)
fig, axes = plt.subplots(
    1, 3, figsize=(15, 4), sharex=True, sharey=True, layout="constrained"
)
for ax, error, name in zip(axes, errors, candidate_names):
    points = ax.scatter(
        validation["item_count"], error,
        c=validation["fragile_count"], cmap="viridis",
        norm=colour_scale, alpha=0.85
    )
    ax.axhline(0, color="grey", linestyle="--")
    ax.set_ylim(-error_limit, error_limit)
    ax.set_title(name)
    ax.set_xlabel("Total items")
axes[0].set_ylabel("Predicted − actual time (minutes)")
fig.colorbar(points, ax=list(axes), label="Fragile items")
plt.show()

**Your interpretation:** ...

### B. Compare MAE and RMSE without changing our selection rule
For absolute errors [2, 2] versus [0, 4], calculate MAE and root mean squared error (RMSE). Both metrics have units of minutes, but RMSE gives larger errors more weight.

Use `np.sqrt(np.mean(errors ** 2))` for RMSE. Why might a business prefer one metric over the other? Do not choose a metric just because it makes your favourite model win.

In [ ]:
# Optional workspace: compare MAE and RMSE for the two error arrays.

### C. Inspect learned coefficients
For the two-feature regression fitted on the original training split, print `model_two.intercept_` and `model_two.coef_`. Match coefficients to `features_two`.

Compare them with the generation formula. Why are they not exactly 2, 0.5, and 1.5? Why should the items-only slope not necessarily equal 0.5?

In [ ]:
# Optional workspace: inspect coefficients.

**Your explanation:** ...

### Reference links
- [scikit-learn: train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [scikit-learn: DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)
- [scikit-learn: LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [scikit-learn: mean_absolute_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)